# 05 — Supervisor Multi-Agent System

**Model:** DeepSeek-V3 via Nebius AI Studio  
**Pattern:** Supervisor / Worker Orchestration  
**Difficulty:** ⭐⭐⭐⭐☆

---

## The Problem

Complex real-world tasks rarely fit a single specialization. Writing a market research report requires: web research, data analysis, clear writing, and quality editing. Cramming all these roles into one agent produces mediocre results across all of them.

**Supervisor Multi-Agent** systems solve this by assembling a team:
- A **Supervisor** that understands the goal, routes tasks to the right specialist, and decides when the job is done.
- **Worker agents**, each optimized for a specific capability (research, analysis, writing).
- Clear handoff protocols so work accumulates toward the final deliverable.

## Architecture

```
┌──────────────────────────────────────────────────────────┐
│               Supervisor Multi-Agent Graph                │
│                                                           │
│              ┌──────────────┐                            │
│   START ────▶│  Supervisor  │◀──────────────────────┐   │
│              └──────┬───────┘                       │   │
│                     │ routes to                     │   │
│         ┌───────────┼───────────┐                   │   │
│         ▼           ▼           ▼                   │   │
│    ┌─────────┐ ┌─────────┐ ┌─────────┐             │   │
│    │Research │ │Analyst  │ │ Writer  │─────────────-┘   │
│    │ Agent   │ │ Agent   │ │ Agent   │                  │
│    └─────────┘ └─────────┘ └─────────┘                  │
│                                           → FINISH → END │
└──────────────────────────────────────────────────────────┘
```

## Setup

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, MessagesState, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict, List, Literal, Annotated
import operator
from tavily import TavilyClient
from pydantic import BaseModel

llm = ChatNebius(
    model="deepseek-ai/DeepSeek-V3",
    temperature=0,
    api_key=os.environ["NEBIUS_API_KEY"]
)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
print("Setup complete.")

## State Design

We use a shared state with `Annotated[List, operator.add]` so each worker **appends** to the message history rather than overwriting it.

In [ ]:
class TeamState(TypedDict):
    messages: Annotated[List, operator.add]   # Full conversation history
    task: str                                  # The original task
    next_agent: str                            # Who the supervisor routes to next
    work_log: Annotated[List[str], operator.add]  # Summary of completed work

## Supervisor Routing

The supervisor uses structured output to reliably choose the next worker. This avoids parsing errors.

In [ ]:
WORKERS = ["researcher", "analyst", "writer", "FINISH"]

class SupervisorDecision(BaseModel):
    next: Literal["researcher", "analyst", "writer", "FINISH"]
    instructions: str  # What specifically the chosen agent should do


SUPERVISOR_SYSTEM = """
You are a team supervisor orchestrating a multi-agent research and writing team.

Your team:
- researcher: gathers current facts and data from the web
- analyst: interprets data, identifies patterns, draws conclusions
- writer: produces polished prose from the gathered information

Given the task and conversation history, decide who should act next and what they should do.
When the final written output is complete, return FINISH.

Be efficient — avoid redundant steps. Move toward FINISH as soon as quality work is done.
"""

supervisor_llm = llm.with_structured_output(SupervisorDecision)


def supervisor_node(state: TeamState) -> dict:
    history = "\n".join(
        f"{m.__class__.__name__}: {m.content[:200]}" for m in state["messages"]
    )
    decision: SupervisorDecision = supervisor_llm.invoke([
        SystemMessage(content=SUPERVISOR_SYSTEM),
        HumanMessage(content=f"Task: {state['task']}\n\nWork so far:\n{history}")
    ])
    print(f"\n[Supervisor] → {decision.next}: {decision.instructions[:100]}")
    return {
        "next_agent": decision.next,
        "messages": [AIMessage(content=f"Supervisor→{decision.next}: {decision.instructions}")]
    }

## Worker Agents

In [ ]:
@tool
def search_web(query: str) -> str:
    """Search the web for facts, statistics, or news."""
    results = tavily.search(query=query, max_results=3)
    return "\n\n".join(r["content"] for r in results.get("results", []))


search_tool_node = ToolNode([search_web])
researcher_llm = llm.bind_tools([search_web])


def researcher_node(state: TeamState) -> dict:
    """Research agent: searches for data and returns findings."""
    last_instruction = state["messages"][-1].content
    print(f"\n[Researcher] Working on: {last_instruction[:80]}...")

    msgs = [
        SystemMessage(content="You are a factual research agent. Use web_search to find current, specific data. Return a clear summary of your findings."),
        HumanMessage(content=last_instruction)
    ]

    for _ in range(4):
        response = researcher_llm.invoke(msgs)
        msgs.append(response)
        if not (hasattr(response, "tool_calls") and response.tool_calls):
            break
        from langchain_core.messages import ToolMessage
        for tc in response.tool_calls:
            result = search_web.invoke(tc["args"])
            msgs.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    result_content = f"[Research Findings]\n{response.content}"
    print(f"   Found: {response.content[:120]}...")
    return {"messages": [AIMessage(content=result_content)], "work_log": ["Research completed"]}


def analyst_node(state: TeamState) -> dict:
    """Analyst agent: interprets data and draws conclusions."""
    print(f"\n[Analyst] Analyzing findings...")
    history = "\n".join(m.content for m in state["messages"] if "Research Findings" in m.content)

    response = llm.invoke([
        SystemMessage(content="You are a data analyst. Extract key patterns, compare figures, and draw actionable conclusions from research data."),
        HumanMessage(content=f"Task: {state['task']}\n\nResearch data:\n{history}\n\nProvide your analysis:")
    ])

    result_content = f"[Analysis]\n{response.content}"
    print(f"   Analysis: {response.content[:120]}...")
    return {"messages": [AIMessage(content=result_content)], "work_log": ["Analysis completed"]}


def writer_node(state: TeamState) -> dict:
    """Writer agent: turns research and analysis into a polished report."""
    print(f"\n[Writer] Composing final report...")
    all_work = "\n\n".join(m.content for m in state["messages"] if "[" in m.content)

    response = llm.invoke([
        SystemMessage(content="You are a professional writer. Produce clear, well-structured prose. Use headers, avoid jargon, and lead with the most important findings."),
        HumanMessage(content=f"Task: {state['task']}\n\nMaterial to write from:\n{all_work}\n\nWrite the final report:")
    ])

    result_content = f"[Final Report]\n{response.content}"
    print(f"   Draft ready.")
    return {"messages": [AIMessage(content=result_content)], "work_log": ["Report written"]}


def route_from_supervisor(state: TeamState) -> str:
    return state["next_agent"]

## Build the Graph

In [ ]:
builder = StateGraph(TeamState)

builder.add_node("supervisor", supervisor_node)
builder.add_node("researcher", researcher_node)
builder.add_node("analyst", analyst_node)
builder.add_node("writer", writer_node)

builder.set_entry_point("supervisor")
builder.add_conditional_edges(
    "supervisor",
    route_from_supervisor,
    {"researcher": "researcher", "analyst": "analyst", "writer": "writer", "FINISH": END}
)

# Each worker reports back to supervisor after completing their task
for worker in ["researcher", "analyst", "writer"]:
    builder.add_edge(worker, "supervisor")

graph = builder.compile()
print("Supervisor multi-agent graph compiled.")

## Live Demo

**Scenario:** Produce a short market snapshot requiring research, analysis, and professional writing.

In [ ]:
task = (
    "Write a concise (300 word) report on the current state of AI chip market competition "
    "in 2025 — focusing on NVIDIA, AMD, and Intel's positions, recent developments, and what "
    "it means for enterprise AI buyers."
)

result = graph.invoke({
    "messages": [HumanMessage(content=task)],
    "task": task,
    "next_agent": "",
    "work_log": []
})

In [ ]:
# Find and print the final report
for msg in reversed(result["messages"]):
    if "[Final Report]" in msg.content:
        print(msg.content)
        break

print("\nWork log:", result["work_log"])

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Structured routing** | Supervisor uses `with_structured_output` — no string parsing |
| **Additive state** | `Annotated[List, operator.add]` prevents messages from being overwritten |
| **Worker specialization** | Each agent has its own system prompt and tool set |
| **Flexible termination** | Supervisor decides FINISH — not a fixed step count |

## What's Next

**Notebook 06** pivots to memory — giving an agent the ability to remember past interactions using a vector store for episodic recall.